In [1]:
print("a")  

a


In [2]:
import mappy as mp
aligner = mp.Aligner("../../data/reference_genome_index.mmi")

In [5]:
from datasets import load_dataset
dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")

/home/pipev/miniconda3/envs/convnova/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Resolving data files: 100%|██████████| 18/18 [00:00<00:00, 35628.82it/s]


In [6]:
filtered = dataset.filter(lambda x: x["task"] == "splice_sites_all")

In [7]:
from Bio import SeqIO
from src.utils.canonicalization import load_ontology, gff_records_generator, parse_record_into_segments_and_skew
from pathlib import Path
from intervaltree import IntervalTree

# Use SeqIO.index for lazy fasta access (does not load full sequences into memory)
fasta_path = "../../data/hg38/ncbi_dataset/data/GCF_000001405.26/GCF_000001405.26_GRCh38_genomic.fna"
training_genome = SeqIO.index(fasta_path, "fasta")
ncbi_to_training_id = {x: x for x in training_genome.keys()}
sequence_name_to_obo_term = load_ontology("../../so.clean.obo")
gff_path = Path("../../data/hg38/ncbi_dataset/data/GCF_000001405.26/genomic.gff")

/home/pipev/miniconda3/envs/convnova/lib/python3.10/site-packages/lightning_lite/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [10]:
training_genome_canonicalization_segments = {}  # cache: id -> { 'positive': IntervalTree, 'negative': IntervalTree }
training_genome_all_features = {}

def _index_features(features, tree):
    """Recorre recursivamente features y sub_features, añadiéndolos al tree."""
    for feat in features:
        s, e = int(feat.location.start), int(feat.location.end)
        if s < e:                          # IntervalTree exige s < e
            gene = (feat.qualifiers.get("gene")
                    or feat.qualifiers.get("Name")
                    or feat.qualifiers.get("ID")
                    or ["?"])[0]
            tree[s:e] = {
                "type":   feat.type,
                "gene":   gene,
                "strand": feat.location.strand,   # +1 / -1 / 0
            }
        sub = getattr(feat, "sub_features", None) or feat.qualifiers.get("_sub_features", [])
        if sub:
            _index_features(sub, tree)


# ── Lazy loader (construye las dos cachés en un único scan del GFF) ────────────
def get_segments_for_id(training_genome_id):
    if training_genome_id in training_genome_canonicalization_segments:
        return training_genome_canonicalization_segments[training_genome_id]

    for record in gff_records_generator(gff_path, {training_genome_id: training_genome_id}):
        tg_id = ncbi_to_training_id.get(record.id, record.id)

        # — árbol de canonicalización (tu código original) —
        pos_list, neg_list = parse_record_into_segments_and_skew(
            record, training_genome, training_genome_id, sequence_name_to_obo_term,
            orient_pseudogene_exons=True, raise_error_if_encounter=("CDS",),
            orient_whole_pseudogene=False, orient_cdjv_segments=True,
        )
        pos_tree = IntervalTree.from_tuples(pos_list) if pos_list else IntervalTree()
        neg_tree = IntervalTree.from_tuples(neg_list) if neg_list else IntervalTree()
        training_genome_canonicalization_segments[training_genome_id] = {
            "positive": pos_tree, "negative": neg_tree
        }

        # — árbol de todos los features (nuevo) —
        all_feat_tree = IntervalTree()
        _index_features(record.features, all_feat_tree)
        training_genome_all_features[training_genome_id] = all_feat_tree

        return training_genome_canonicalization_segments[training_genome_id]

    # no encontrado en GFF
    training_genome_canonicalization_segments[training_genome_id] = {
        "positive": IntervalTree(), "negative": IntervalTree()
    }
    training_genome_all_features[training_genome_id] = IntervalTree()
    return training_genome_canonicalization_segments[training_genome_id]

In [11]:
from collections import Counter

for seq in filtered['train']['sequence'][3:]:
    hit = next((x for x in aligner.map(seq) if x.is_primary), None)

    if hit is None:
        raise RuntimeError(f"No primary hit for sequence #{total}")
    hit_id = ncbi_to_training_id.get(hit.ctg, hit.ctg)
    # lazily build/get IntervalTrees for this hit id
    segs = get_segments_for_id(hit_id)
    if hit.strand == 1:
        tree = segs['positive']
    elif hit.strand == -1:
        tree = segs['negative']
    else:
        raise ValueError('Invalid strand: {}'.format(hit.strand))
    overlaps = tree.overlap(hit.r_st, hit.r_en)
    if not overlaps:
        # ── diagnóstico: qué feature hay realmente en esa región ──────────────
        all_feat_tree = training_genome_all_features.get(hit_id, IntervalTree())
        feat_hits = all_feat_tree.overlap(hit.r_st, hit.r_en)

        if feat_hits:
            type_counts = Counter(iv.data["type"] for iv in feat_hits)
            genes       = sorted({iv.data["gene"] for iv in feat_hits})
            same_strand = [iv for iv in feat_hits if iv.data["strand"] == hit.strand]
            print(
                f"[MISS] {hit_id}:{hit.r_st}-{hit.r_en} strand={hit.strand:+d} "
                f"| feature types: {dict(type_counts)} "
                f"| genes: {genes} "
                f"| same-strand hits: {len(same_strand)}/{len(feat_hits)}"
            )
        else:
            print(
                f"[MISS] {hit_id}:{hit.r_st}-{hit.r_en} strand={hit.strand:+d} "
                f"→ intergénico (sin features GFF en esa región)"
            )

        raise

[MISS] NC_000006.12:9092016-9092416 strand=+1 | feature types: {'region': 1} | genes: ['6'] | same-strand hits: 1/1


RuntimeError: No active exception to reraise